# **Results:**

## Problem 1:

Here calculations are performed using Historical Simulation (HS) and Parametric method.

Metric | Value ($)
-------------------|------------------
EL (Expected payout) — Parametric | 377705
EL (Expected payout) — Parametric | 377705
P99 (99th percentile payout) — Parametric | 1400000
EL (Expected payout) — Historical | 740000
P99 (99th percentile payout) — Historical (nearest) | 2100000
----

&nbsp;&nbsp;

## Problem 2:

Here calculations are performed only using Historical Simulation (HS) method.

Metric | Value ($)
-------------------|------------------
Winters used| 11
EL  | 3,773,916
P99 (99th percentile payout) — Parametric | 1400000
P99 (nearest)| 10,689,954
P99 (linear) | 10,181,575
----

&nbsp;&nbsp;
&nbsp;&nbsp;

**Note:** Running the 2nd and 4th cell will ask to upload problem files. Please use seperate problem file for problem 1 and 2.
For example - Problem 1 file = Intro (Sheet 1) + ConsecAboveFreezing (Sheet 2)

In [ ]:
import io, sys, math, json, typing, warnings
import numpy as np
import pandas as pd

try:
    # Colab upload utility
    from google.colab import files
    COLAB = True
except Exception:
    COLAB = False

warnings.filterwarnings('ignore')

def p99_nearest(x: np.ndarray, q=0.99):
    x = np.asarray(x).astype(float)
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return np.nan
    x_sorted = np.sort(x)
    k = int(np.ceil(q * len(x_sorted)))
    k = max(1, min(k, len(x_sorted)))
    return float(x_sorted[k-1])

def ensure_datetime_series(df, col):
    df[col] = pd.to_datetime(df[col])
    return df

def feb_last_day(year:int)->int:
    return 29 if ((year % 4 == 0 and year % 100 != 0) or (year % 400 == 0)) else 28

def consecutive_pairs_count(dates: pd.Series, tmin: pd.Series, threshold=32.0) -> int:
    s = pd.DataFrame({'Date': pd.to_datetime(dates), 'TMin': tmin}).sort_values('Date')
    s['I'] = (s['TMin'] >= threshold).astype(int)
    s['lagI'] = s['I'].shift(1)
    s['dt'] = (s['Date'] - s['Date'].shift(1)).dt.days
    s['pair'] = np.where(s['dt']==1, s['I']*s['lagI'], 0)
    return int(s['pair'].sum())

def quant_summary(x):
    return {
        'EL': float(np.mean(x)),
        'P50': float(np.quantile(x, 0.50)),
        'P95': float(np.quantile(x, 0.95)),
        'P99_linear': float(np.quantile(x, 0.99)),
        'P99_nearest': float(p99_nearest(x, 0.99))
    }


In [ ]:
print("=== Problem 1: Upload workbook (e.g., assignment1_Pricing Problems_1.xlsx) ===")
if COLAB:
    uploaded = files.upload()
    assert len(uploaded)==1, "Please upload exactly one file for Problem 1."
    p1_fname = list(uploaded.keys())[0]
else:
    p1_fname = input("Local path to Problem 1 Excel file: ").strip()
print("Using file:", p1_fname)


=== Problem 1: Upload workbook (e.g., assignment1_Pricing Problems_1.xlsx) ===


Saving assignment1_Pricing Problems_1.xlsx to assignment1_Pricing Problems_1.xlsx
Using file: assignment1_Pricing Problems_1.xlsx


In [ ]:
# Read the sheets we need
xls = pd.ExcelFile(p1_fname)
sheet_candidates = [s for s in xls.sheet_names if 'Consec' in s or 'Above' in s or s.lower().startswith('consec')]
sheet_p1 = sheet_candidates[0] if sheet_candidates else xls.sheet_names[0]
df = pd.read_excel(p1_fname, sheet_name=sheet_p1)

# Basic normalization
for c in df.columns:
    if 'date' in str(c).lower():
        date_col = c
        break
else:
    raise ValueError("Couldn't find Date column in Problem 1 sheet")

col_tmax = next((c for c in df.columns if str(c).lower().startswith('tmax')), None)
col_tmin = next((c for c in df.columns if str(c).lower().startswith('tmin')), None)
assert col_tmin is not None, "Missing TMin column"

df = df[[date_col, col_tmin] + ([col_tmax] if col_tmax else [])].copy()
df.rename(columns={date_col:'Date', col_tmin:'TMin'}, inplace=True)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').dropna(subset=['TMin']).reset_index(drop=True)

# Risk window
start = pd.Timestamp(2015,12,1); end = pd.Timestamp(2016,2,29)

# Historical Simulation over winters
def winter_pairs_for_year(y):
    s = pd.Timestamp(y,12,1); e = pd.Timestamp(y+1,2,feb_last_day(y+1))
    d = df[(df['Date']>=s) & (df['Date']<=e)]
    if len(d) < 85:
        return None
    return consecutive_pairs_count(d['Date'], d['TMin'], threshold=32.0)

years = sorted(df['Date'].dt.year.unique())
pairs_list = []
for y in years:
    val = winter_pairs_for_year(y)
    if val is not None:
        pairs_list.append({'WinterStart': y, 'Pairs': int(val)})

hs_p1 = pd.DataFrame(pairs_list).sort_values('WinterStart')

# Ex-ante distribution excludes the target winter by convention
hs_ex_ante = hs_p1[hs_p1['WinterStart'] != 2015]

EL_hs_pairs = float(hs_ex_ante['Pairs'].mean())
P99_hs_pairs_nearest = float(p99_nearest(hs_ex_ante['Pairs'].to_numpy(), 0.99))
P99_hs_pairs_linear  = float(np.quantile(hs_ex_ante['Pairs'].to_numpy(), 0.99))

print("\n[Problem 1 — Historical Simulation]")
print(f"Winters used: {len(hs_ex_ante)}")
print(f"EL (pairs): {EL_hs_pairs:.3f}")
print(f"P99 (pairs, nearest): {P99_hs_pairs_nearest:.3f}")
print(f"P99 (pairs, linear):  {P99_hs_pairs_linear:.3f}")

# Parametric Seasonal + AR(1) Monte Carlo (toggle)
USE_PARAMETRIC = True
if USE_PARAMETRIC:
    # Fit Fourier seasonal mean with 2 harmonics to daily TMin
    tmin = df.set_index('Date').asfreq('D')['TMin'].astype(float)
    tmin = tmin.dropna()
    idx = tmin.index
    doy = idx.dayofyear.to_numpy().astype(float)
    doy = np.where(doy==366, 365, doy)
    X = [np.ones_like(doy), np.sin(2*np.pi*doy/365), np.cos(2*np.pi*doy/365),
         np.sin(4*np.pi*doy/365), np.cos(4*np.pi*doy/365)]
    X = np.vstack(X).T
    beta = np.linalg.lstsq(X, tmin.to_numpy(), rcond=None)[0]
    mu_hat = (X @ beta)

    eps = tmin.to_numpy() - mu_hat
    # AR(1) on consecutive days
    # build mask of consecutive days
    daydiff = (idx[1:] - idx[:-1]).days
    mask = daydiff==1
    y = eps[1:][mask]; x = eps[:-1][mask]
    phi = float((x@y)/(x@x))
    eta = y - phi*x
    sigma = float(np.std(eta, ddof=1))
    var_stationary = sigma**2/(1-phi**2)

    # Simulate Dec–Feb
    sim_idx = pd.date_range(start, end, freq='D')
    doy_s = sim_idx.dayofyear.to_numpy().astype(float)
    doy_s = np.where(doy_s==366, 365, doy_s)
    Xs = np.vstack([np.ones_like(doy_s),
                    np.sin(2*np.pi*doy_s/365), np.cos(2*np.pi*doy_s/365),
                    np.sin(4*np.pi*doy_s/365), np.cos(4*np.pi*doy_s/365)]).T
    mu_s = (Xs @ beta)

    n_days = len(sim_idx); n_sims = 20000
    rng = np.random.default_rng(12345)
    pairs_sim = np.empty(n_sims, dtype=int)
    for m in range(n_sims):
        e = np.empty(n_days)
        e[0] = rng.normal(0.0, np.sqrt(var_stationary))
        for t in range(1, n_days):
            e[t] = phi*e[t-1] + rng.normal(0.0, sigma)
        tmin_sim = mu_s + e
        I = (tmin_sim >= 32).astype(int)
        pairs_sim[m] = int(np.sum(I[1:]*I[:-1]))
    qs = quant_summary(pairs_sim)
    print("\n[Problem 1 — Parametric Seasonal+AR(1)]")
    for k,v in qs.items():
        print(f"{k}: {v:.3f}")
else:
    print("\nParametric model skipped (USE_PARAMETRIC=False).")

# Dollar mapping helper
def pairs_to_dollars(pairs, notional_per_pair=100000.0):
    return float(pairs*notional_per_pair)

print("\n[Problem 1 — Dollarized (Notional $100k per pair)]")
print(f"EL (HS, $): {pairs_to_dollars(EL_hs_pairs):,.0f}")
print(f"P99 (HS, $, nearest): {pairs_to_dollars(P99_hs_pairs_nearest):,.0f}")
if USE_PARAMETRIC:
    print(f"EL (Parametric, $): {pairs_to_dollars(qs['EL']):,.0f}")
    print(f"P99 (Parametric, $): {pairs_to_dollars(qs['P99_nearest']):,.0f}")



[Problem 1 — Historical Simulation]
Winters used: 35
EL (pairs): 7.400
P99 (pairs, nearest): 21.000
P99 (pairs, linear):  19.300

[Problem 1 — Parametric Seasonal+AR(1)]
EL: 3.777
P50: 3.000
P95: 10.000
P99_linear: 14.000
P99_nearest: 14.000

[Problem 1 — Dollarized (Notional $100k per pair)]
EL (HS, $): 740,000
P99 (HS, $, nearest): 2,100,000
EL (Parametric, $): 377,705
P99 (Parametric, $): 1,400,000


In [ ]:
print("\n=== Problem 2: Upload workbook (e.g., Problem_2.xlsx) ===")
if COLAB:
    uploaded2 = files.upload()
    assert len(uploaded2)==1, "Please upload exactly one file for Problem 2."
    p2_fname = list(uploaded2.keys())[0]
else:
    p2_fname = input("Local path to Problem 2 Excel file: ").strip()
print("Using file:", p2_fname)



=== Problem 2: Upload workbook (e.g., Problem_2.xlsx) ===


Saving Problem_2.xlsx to Problem_2.xlsx
Using file: Problem_2.xlsx


In [ ]:
# Read data sheet
raw = pd.read_excel(p2_fname, sheet_name="Gas-Temp Quanto")

# Gas block
gas_header_row = raw.index[(raw.iloc[:,12] == "Date") & (raw.iloc[:,13] == "Tetco M3")].tolist()
assert gas_header_row, "Couldn't find Gas headers (Date, Tetco M3)."
r_g = gas_header_row[0]
gas_df = raw.iloc[r_g+1:, [12,13]].dropna(how="all")
gas_df.columns = ["Date","Tetco_M3"]
gas_df["Date"] = pd.to_datetime(gas_df["Date"])
gas_df = gas_df.dropna().sort_values("Date").reset_index(drop=True)

# Temp block
temp_header_row = raw.index[(raw.iloc[:,15] == "Dates") & (raw.iloc[:,16] == "TMax") & (raw.iloc[:,17] == "TMin")].tolist()
assert temp_header_row, "Couldn't find Temp headers (Dates, TMax, TMin)."
r_t = temp_header_row[0]
temp_df = raw.iloc[r_t+1:, [15,16,17]].dropna(how="all")
temp_df.columns = ["Date","TMax","TMin"]
temp_df["Date"] = pd.to_datetime(temp_df["Date"])
temp_df = temp_df.dropna().sort_values("Date").reset_index(drop=True)

# Forwards (P_strike)
fwd_rows = raw.iloc[6:20, 19:21].dropna(how="all")
fwd_rows.columns = ["MonthDate","Forward"]
fwd_rows["MonthDate"] = pd.to_datetime(fwd_rows["MonthDate"])
fwd_rows = fwd_rows.dropna().reset_index(drop=True)

target_months = pd.to_datetime(["2015-11-01","2015-12-01","2016-01-01","2016-02-01","2016-03-01"])
fwd_map = dict(zip(fwd_rows["MonthDate"], fwd_rows["Forward"]))
P_strikes = [float(fwd_map[m]) for m in target_months]
T_strikes = [50, 41, 37, 38, 48]
NOTIONAL = 50000.0

# Monthly averages
gas_df["MonthDate"] = gas_df["Date"].values.astype("datetime64[M]")
P_m = gas_df.groupby("MonthDate", as_index=False).agg(P_avg=("Tetco_M3","mean"))

temp_df["T_avg_daily"] = (temp_df["TMax"] + temp_df["TMin"]) / 2.0
temp_df["MonthDate"] = temp_df["Date"].values.astype("datetime64[M]")
T_m = temp_df.groupby("MonthDate", as_index=False).agg(T_avg=("T_avg_daily","mean"))

monthly = pd.merge(T_m, P_m, on="MonthDate", how="inner").sort_values("MonthDate")

# Build HS year blocks
years = sorted(set(monthly["MonthDate"].dt.year))
totals = []
for y in years:
    months_needed = [
        pd.Timestamp(y,11,1), pd.Timestamp(y,12,1),
        pd.Timestamp(y+1,1,1), pd.Timestamp(y+1,2,1), pd.Timestamp(y+1,3,1)
    ]
    sub = monthly[monthly["MonthDate"].isin(months_needed)].copy()
    if len(sub) != 5:
        continue
    sub = sub.set_index("MonthDate").reindex(months_needed).reset_index()
    sub["T_strike"] = T_strikes
    sub["P_strike"] = P_strikes
    sub["T_leg"] = np.maximum(sub["T_strike"] - sub["T_avg"], 0.0)
    sub["P_leg"] = np.maximum(sub["P_avg"] - sub["P_strike"], 0.0)
    sub["Payoff_m"] = NOTIONAL * sub["T_leg"] * sub["P_leg"]
    totals.append(sub["Payoff_m"].sum())

totals = np.array(totals, dtype=float)
print("\n[Problem 2 — Gas–Temp Quanto (HS)]")
print(f"Winters used: {len(totals)}")
print(f"EL ($): {np.mean(totals):,.0f}")
print(f"P99 ($, nearest): {p99_nearest(totals):,.0f}")
print(f"P99 ($, linear):  {np.quantile(totals,0.99):,.0f}")



[Problem 2 — Gas–Temp Quanto (HS)]
Winters used: 11
EL ($): 3,773,916
P99 ($, nearest): 10,689,954
P99 ($, linear):  10,181,575
